# Turkish Medical Agentic RAG

Run this notebook from the `agentic_rag/` directory. It validates the corpus/index, connects Together AI, and exposes a stateful chat helper.

In [ ]:
from dataclasses import asdict
from statistics import median

from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage
from transformers import AutoTokenizer

from scripts.indexing.chunker import ChunkingConfig, chunk_documents
from scripts.indexing.indexer import ensure_index
from scripts.indexing.loader import load_markdown_documents
from scripts.llm.together import create_together_llms
from scripts.retrieval.retriever import load_retriever
from scripts.settings import RagSettings
from scripts.workflow.graph import build_graph

settings = RagSettings.from_env()
settings

## Preview the corpus and chunk contract

The preview chunks one document only; the index step processes all documents.

In [ ]:
documents = load_markdown_documents(settings.data_dir)
tokenizer = AutoTokenizer.from_pretrained(settings.embedding_model)
chunk_config = ChunkingConfig(
    tokenizer_name=settings.embedding_model,
    max_tokens=settings.chunk_size_tokens,
    overlap_tokens=settings.chunk_overlap_tokens,
)
preview_chunks = chunk_documents(documents[:1], chunk_config, tokenizer=tokenizer)
preview_token_counts = [
    len(tokenizer.encode(chunk.page_content, add_special_tokens=False))
    for chunk in preview_chunks
]
{
    'documents': len(documents),
    'preview_source': documents[0].metadata['source'],
    'preview_chunks': len(preview_chunks),
    'preview_median_tokens': median(preview_token_counts),
    'preview_max_tokens': max(preview_token_counts),
}

## Create or validate Qdrant

Leave `FORCE_REINDEX` false unless a stale-index error has been reviewed.

In [ ]:
FORCE_REINDEX = False
index_report = ensure_index(settings, force_recreate=FORCE_REINDEX)
asdict(index_report)

In [ ]:
retriever = load_retriever(settings)
sanity_results = retriever.invoke('demir eksikliği anemisi laboratuvar bulguları')
[
    {
        'source': document.metadata.get('source'),
        'title': document.metadata.get('title'),
        'subtitle': document.metadata.get('subtitle'),
        'chunk_index': document.metadata.get('chunk_index'),
    }
    for document in sanity_results
]

## Build the Together AI graph

In [ ]:
llms = create_together_llms(settings)
graph = build_graph(llms.responder, llms.pruning, retriever)
display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

In [ ]:
conversation = {'messages': []}

def chat(user_input: str) -> str:
    if not user_input.strip():
        raise ValueError('user_input cannot be empty')
    conversation['messages'].append(HumanMessage(content=user_input.strip()))
    result = graph.invoke(conversation)
    conversation['messages'] = result['messages']
    return result['messages'][-1].content


In [ ]:
answer = chat('Demir eksikliği anemisinin laboratuvar bulguları nelerdir?')
display(Markdown(answer))